# AI verklaring



Ik heb GPT5.6 luna gebruikt om te sparren met naar verschillende aspecten zoals welke modellen en prestaties voor betere prestaties zou kunnen zorgen.

Verder heb ik GPT5.6 Luna gebruikt om fouten/defecten te corrigeren (zoals in deployment waarbij webserver niet werkte)

Daarnaast heb ik gevraagd naar de best practices en deze zodanig toegepast dat het realistisch werkbaar was.

# Wat gedaan met feedback

- Ik ben duidelijk stap voor stap gaan werken volgens de lesstof op de ELO. Nu volg ik de handelingen die op ELO staan beschreven alvorens er complexe zaken bij komen kijken.
- Bij elke punt geef ik duidelijk uitleg; obervaties en beredeneer ik mijn keuzes grondig
- Als ik ergens vastloop schakel ik vroegtijdig bij
- Bij onduidelijkheden stel ik vragen aan de docenten

Ten eerste zie je dat ik echt grondig de data ben gaan kijken zoals in deze opdracht (bij cosine-similarities) hier heb ik echt vergeleken op best presteerende modellen alvorens ik de interface en chatbot maakte.

Ten tweede heb ik bij elke punt nu duidelijk uitleg gegeven; observaties van wat ik zie en wat ik nu ga doen. 

Nu ben ik niet te snel gaan werken naar het eind punt maar heb ik dit geleidelijk opgepakt naar het juiste resultaat. 


# CRISP-DM Iteratie 1

## Business Understanding

Studenten vinden het onderwijs- en examenreglement (OER) moeilijk toegankelijk en bestuderen het daardoor onvoldoende. Het doel is een eenvoudige Nederlandstalige chatbot die vragen over het OER beantwoordt. De chatbot mag geen nieuwe feiten verzinnen: als het antwoord niet in de opgehaalde bron staat, moet dat duidelijk worden aangegeven.

### KPI's
- Minimaal 80% van de antwoorden is correct.
- Minimaal 4 van de 5 opgehaalde fragmenten zijn relevant.
- Antwoorden bevatten een bronpagina.
- De chatbot verzint geen informatie.
- Een antwoord verschijnt binnen 60 seconden.

### Kritische succesfactoren (KSF's)
- De OER wordt correct ingelezen en gechunked.
- Het model antwoordt alleen op basis van de context.
- De bronverwijzing is duidelijk.
- De chatbot werkt stabiel tijdens de demo.

### Ranvoorwaarde

- Context is beperkt tot hooguit 1500 tokens per vraag



## Data Understanding

De kennisbron is het OER PDF. Dit document bevat de opleidingsspecifieke regels voor studenten.

In deze opdracht is er geen sprake van een dataset, het toegang hebben tot specifiek genoemd document (OER) is al prima. 

We gaan er vanuit dat deze OER document inhoudelijk goed is en aan de vereisten voldoet.

Ethiek: in hoeverre kan de chatbot verantwoord gesteld voor onethische informatie uit het OER?

## Data Preparation

In deze fase gaan we data: chunken, embedden en opslaan en een juiste start model kiezen.


Eerst de benodigde pdf lezer installeren:

```python
%pip install pypdf chromadb keras-nlp
```

Nu beginnen we eerst met een LLM. Net zoals op de kennisbasis omschreven staat. Zelf vond ik het handiger eerst data te chunken, embedden en opslaan voordat het model gekozen wordt, maar kennisbasis is leidend.



In [1]:
# volgens kennisbasis
from sentence_transformers import SentenceTransformer, util

# 1. Laad een voorgetraind embedding-model
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Documenten uit de OER en een vraag van een student
documenten = [
    "Je schrijft je uit voor een vak via Studielink.",
    "Het herkansingsbeleid en het aantal toetskansen staan in de OER.",
    "De bibliotheek is open van 8 tot 22 uur.",
]
vraag = "Hoeveel kansen heb ik om een tentamen te halen?"

# 3. Zet zowel de documenten als de vraag om naar vectoren
doc_vectoren = model.encode(documenten)
vraag_vector = model.encode(vraag)

# 4. Bereken de cosine-similarity tussen de vraag en elk document
scores = util.cos_sim(vraag_vector, doc_vectoren)
print(scores)

/Users/irshad/miniconda3/envs/AI_2526_py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1117 17:37:07.394000 3052 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12144.76it/s]


tensor([[0.5316, 0.5050, 0.3716]])


We testen eerst of het model al goed genoeg semantisch linken kan leggen. Maar dat blijkt dus niet. 0.53 is het hoogste voorspelde waarde (die ook nog eens fout ligt) we zouden eigenlijk de middelste waarde verwachten maar die heeft 0.50. Nog lager dan verwacht.



We testen nu een ander model dat wel geschikt is voor soortgelijke cosine-similarities. Er werd paraphrase-multilingual aangeraden.

In [2]:
# We testen nu een ander model dat wel geschikt is voor soortgelijke cosine-similarities. 

from sentence_transformers import SentenceTransformer, util

# 1. Laad een multilingual embedding-model dat ook Nederlands ondersteunt
model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

# 2. Gebruik precies dezelfde documenten en vraag als hierboven
documenten = [
    "Je schrijft je uit voor een vak via Studielink.",
    "Het herkansingsbeleid en het aantal toetskansen staan in de OER.",
    "De bibliotheek is open van 8 tot 22 uur.",
]
vraag = "Hoeveel kansen heb ik om een tentamen te halen?"

# 3. Zet documenten en vraag om naar vectoren
doc_vectoren = model.encode(documenten)
vraag_vector = model.encode(vraag)

# 4. Bereken de cosine-similarity
scores = util.cos_sim(vraag_vector, doc_vectoren)
print(scores)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8239.06it/s]


tensor([[0.3495, 0.4366, 0.2295]])


We zien hier dat de resultaat nog niet heel doorslag gevend ligt, maar dat hij nu wel de hoogste waarde op de juiste manier voorspelt. Maar nog niet helemaal zoals we dat gewilt hadden. We gaan in ieder geval verder met deze model.  Wat belangrijk om te weten is dat dit een embedding-model is, we zullen dit combineren met een ander model. 

Ethiek: hoe is het ethisch verantwoord dat wij nu middels cosine similarities bekijken naar meetbaarheid terwijl dat soms niet uitmaakt? Soms zien we dat cosine similarities laag is maar het antwoord nauwkeurig. 

We gaan nu de data chunken, embedden en opslaan. 

In [3]:
# 1. Lees de PDF en maak tekstchunks
from pathlib import Path
import re

import chromadb
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

pdf_path = Path("Opleidingsdeel OER Bachelor ICT voltijd Almere 2026-2027.pdf")
reader = PdfReader(str(pdf_path))
if reader.is_encrypted:
    reader.decrypt("")

chunks = []
for page_number, page in enumerate(reader.pages, start=1):
    text = re.sub(r"\s+", " ", page.extract_text() or "").strip()
    words = text.split()
    for start in range(0, len(words), 150):
        chunk = " ".join(words[start:start + 180])
        if len(chunk) >= 80:
            chunks.append({"text": chunk, "page": page_number})
        if start + 180 >= len(words):
            break

print(f"1. {len(chunks)} chunks gemaakt")

# 2. Embed alle chunks met het gekozen Nederlandstalige model
embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
texts = [chunk["text"] for chunk in chunks]
embeddings = embedding_model.encode(texts)
print(f"2. {len(embeddings)} embeddings gemaakt")

# 3. Sla de chunks, embeddings en paginanummers op in ChromaDB
client = chromadb.Client()
collection = client.get_or_create_collection(
    "oer_minimal_embeddings_cosine", metadata={"hnsw:space": "cosine"}
)
collection.upsert(
    ids=[f"chunk-{i}" for i in range(len(chunks))],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[{"page": chunk["page"]} for chunk in chunks],
)
print(f"3. {collection.count()} chunks opgeslagen in ChromaDB")

# 4. Test één vraag en toon het beste gevonden resultaat
vraag = "Hoeveel toetskansen heb ik?"
vraag_embedding = embedding_model.encode([vraag])[0].tolist()
resultaat = collection.query(query_embeddings=[vraag_embedding], n_results=1)
print(f"\nTestvraag: {vraag}")
print(f"Pagina: {resultaat['metadatas'][0][0]['page']}")
print(f"Resultaat: {resultaat['documents'][0][0][:300]}...")
print(f"Cosine-similarity: {1 - resultaat['distances'][0][0]:.4f}")


1. 77 chunks gemaakt


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 24948.93it/s]


2. 77 embeddings gemaakt
3. 77 chunks opgeslagen in ChromaDB

Testvraag: Hoeveel toetskansen heb ik?
Pagina: 29
Resultaat: 4 • Toetsweek: 28 juni – 2 juli 2027 5.2 Extra kans propedeuse toetsen / versnellen • Schriftelijke toetsen: 9 – 14 aug 2027 • Assessments/producten: 16 – 29 aug 2027 Deze periode is bedoeld voor: • Studenten met 40–59 studiepunten in het 1e jaar van inschrijving in de propedeuse fase (3e kans). Bij...
Cosine-similarity: 0.6205


**Observaties**

We hebben hier de vraag omgezet naar embedding/vector, chromaDB gebruikt die vectoren om te vergelijken met alle pdf-chunks, de teksten worden opgehaald en de bijbehorende paginanummer wordt getoond.

Er zijn 77 chunks gemaakt. Het aantal chunks dat gebruikt wordt heeft deels ook invloed in hoe goed een model antwoord geeft op vragen. We testen hiermee, daarna zulen wij indien nodig ook system prompt/user prompts fine-tunen. 

ChromaDB kan inmiddels al de relevante vectoren bij elkaar zetten. Zoals paginanummer, resultaat etc. Ik heb nu de LLM nodig die we in voorgaande hoofdstuk bespraken om er een menselijk antwoord op terug te geven. De cosine similarity is ook vrij hoog 0.62, dat moeten we nog beproeven of dat daadwerkelijk goed genoeg is. 

We gaan nu dus verder met RAG (Retrieve, Augmenteren, Genereren) in Modeling. We hebben nu 77 chunks, 77 embeddings en 77 chromadb items

# Modeling


**Observaties**

In de vorige stap zagen we dat paraphrase-multilingual betere waardes gaf. Maar dit model kan geen tekst genereren. Het is een embeddingmodel. Voor het genereren van een antwoord gebruiken we Phi-3 model.

Dus paraphrase-multilingual > embedding en retrieval > Phi-3 > Antwoord geven


**Update**

Middels de Phi-3 zag ik dat de laptop crashte door te veel RAM gebruik. Na wat onderzoek heb ik zelf een LLM gehost: de qwen model met 4B parameters. Dit is een GGUF model die veel minder geheugen gebruikt


Dus nu ziet het er als volgt uit: Paraphrase: embedding en retrieval > Qwen: antwoord geven 

Qwen draait lokaal


In [4]:
# 1. Retrieve: zoek meerdere relevante chunks bij de vraag.
vraag = "Hoeveel toetskansen heb ik?"
vraag_embedding = embedding_model.encode([vraag])[0].tolist()
resultaat = collection.query(
    query_embeddings=[vraag_embedding],
    n_results=5,
)

bronnen = [
    {"tekst": tekst, "pagina": metadata["page"]}
    for tekst, metadata in zip(
        resultaat["documents"][0], resultaat["metadatas"][0]
    )
]
print(f"1. Retrieve klaar: {len(bronnen)} relevante chunks gevonden.")
for bron, afstand in zip(bronnen, resultaat["distances"][0]):
    print(
        f"Pagina {bron['pagina']} - cosine-similarity: {1 - afstand:.4f}"
    )

# 2. Augmenteren: voeg de gevonden chunks toe aan de prompt.
context = "\n\n".join(
    f"[Pagina {bron['pagina']}] {bron['tekst']}" for bron in bronnen
)
prompt = f"""Je bent een nauwkeurige Nederlandstalige OER-assistent.
Beantwoord uitsluitend de vraag en gebruik alleen de gegeven context.
Geef een kort en concreet antwoord in maximaal vier zinnen.
Neem voorwaarden, uitzonderingen en doelgroepen uit de context mee.
Verzin niets en trek geen conclusies die niet letterlijk of duidelijk uit de context volgen.
Als de context het antwoord niet bevat, zeg dan: 'Dit kan ik niet vinden in het OER.'
Vermeld aan het einde de gebruikte bronpagina's.

Context:
{context}

Vraag: {vraag}"""
print("2. Augmenteren klaar: context aan prompt toegevoegd.")

# 3. Genereren: laat Qwen een antwoord maken via Ollama.
# Ollama gebruikt een compact GGUF-model en daardoor veel minder geheugen.
import json
from urllib.request import Request, urlopen

ollama_request = Request(
    "http://localhost:11434/api/chat",
    data=json.dumps({
        "model": "qwen2.5:3b",
        "messages": [
            {"role": "system", "content": "Je bent een nauwkeurige OER-assistent. Geef geen informatie die niet in de context staat."},
            {"role": "user", "content": prompt},
        ],
        "stream": False,
        "options": {"temperature": 0.1},
    }).encode("utf-8"),
    headers={"Content-Type": "application/json"},
)
with urlopen(ollama_request, timeout=300) as response:
    antwoord = json.loads(response.read().decode("utf-8"))["message"]["content"].strip()
print("3. Genereren klaar:\n")
print(antwoord)

1. Retrieve klaar: 5 relevante chunks gevonden.
Pagina 29 - cosine-similarity: 0.6205
Pagina 28 - cosine-similarity: 0.5648
Pagina 25 - cosine-similarity: 0.5212
Pagina 14 - cosine-similarity: 0.4997
Pagina 19 - cosine-similarity: 0.4732
2. Augmenteren klaar: context aan prompt toegevoegd.
3. Genereren klaar:

Je hebt toetskansen voor de volgende periode van toetsen van 9 tot 14 augustus 2027. Dit is bedoeld voor studenten met 40-59 studiepunten in het eerste jaar van de propedeuse fase.


Ethiek: in hoeverre is het ethisch verantwoord om een model te hosten die niet volledig getrained is binnen het bedrijf maar wel nog alleen met interne bedrijfsdata moet werken?

# Evaluation



We zien dat het model nog niet adequaat antwoord kan geven. Zo wijk het antwoord vaak af. We zullen wat optimimalisaties in het RAG toevoegen. De vraag die hierboven werd geantwoord werd opzich niet verkeerd geantwoord maar het kan wel wat optimalisatie gebruiken natuurlijk. 

We zien dat het model opzich goed antwoord kan geven op de vraag met 5 gevonden relevante chunks. 

We zullen na wat optimalisaties testen met meerdere vragen (dit is ons meetlat)


# CRISP-DM Iteratie 2



We gaan wat optimalisaties toevoegen. in eerste instantie op retrieval:

- We halen drie tot vijf relevante chunks op.
- We behouden het paginanummer als bronverwijzing.
- We controleren of de belangrijkste informatie in de opgehaalde context staat.
- We gebruiken het multilingual embeddingmodel voor Nederlandse vragen en OER-tekst.
- We gebruiken Qwen-3b model lokaal

Hiernaast gaan we de system en user prompt aanscherpen. Zodat Qwen een kort en duidelijk antwoord geeft




 Middels deze stappen gaan we testen hoe goed het model nu eigenlijk is. Beetje vergelijkbaar als hoe je val_acc en acc test. Maar hier hebben wij geen leidraad gekregen verder dan cosine-similarities. Daarom wordt er hier handmatig vergeleken middels enkele vragen

In [5]:
# Middels deze stappen gaan we testen hoe goed het model nu eigenlijk is. Beetje vergelijkbaar als hoe je val_acc en acc test. Maar hier hebben wij geen leidraad gekregen verder dan cosine-similarities. Daarom wordt er hier handmatig vergeleken middels enkele vragen

 
# 1. Retrieven
vraag = "1. Wat kun je mij vertellen over stage?"
#vragen in array doen 
vraag_embedding = embedding_model.encode([vraag])[0].tolist()

resultaat = collection.query(
    query_embeddings=[vraag_embedding],
    n_results=8,
)

bronnen = [
    {"tekst": tekst, "pagina": metadata["page"]}
    for tekst, metadata in zip(
        resultaat["documents"][0],
        resultaat["metadatas"][0],
    )
]

print(f"Retrieve klaar: {len(bronnen)} chunks gevonden.")
for bron, afstand in zip(bronnen, resultaat["distances"][0]):
    print(
        f"Pagina {bron['pagina']} | "
        f"cosine-similarity: {1 - afstand:.4f}"
    )

# 2. Augmenteren: voeg de bronnen toe aan de prompt
context = "\n\n".join(
    f"[Pagina {bron['pagina']}] {bron['tekst']}"
    for bron in bronnen
)

prompt = f"""Beantwoord de vraag over het OER in duidelijk Nederlands.

Regels:
- Gebruik uitsluitend de informatie uit de context.
- Geef maximaal vier zinnen.
- Neem relevante voorwaarden en uitzonderingen mee.
- Verzin niets en trek geen conclusies die niet uit de context volgen.
- Als de context het antwoord niet bevat, zeg dan exact:
  "Dit kan ik niet vinden in het OER."
- Vermeld aan het einde de gebruikte bronpagina's.

Context:
{context}

Vraag: {vraag}
"""

# 3. Genereren met Qwen via Ollama
ollama_request = Request(
    "http://localhost:11434/api/chat",
    data=json.dumps({
        "model": "qwen2.5:3b",
        "messages": [
            {
                "role": "system",
                "content": (
                    "Je bent een nauwkeurige Nederlandstalige OER-assistent. "
                    "Gebruik uitsluitend de verstrekte context. "
                    "Verzin nooit informatie. "
                    "Volg alle antwoordregels uit de gebruikersprompt."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        "stream": False,
        "options": {"temperature": 0.1},
    }).encode("utf-8"),
    headers={"Content-Type": "application/json"},
)

with urlopen(ollama_request, timeout=300) as response:
    antwoord = json.loads(
        response.read().decode("utf-8")
    )["message"]["content"].strip()

print("\nAntwoord:\n")
print(antwoord)

Retrieve klaar: 8 chunks gevonden.
Pagina 17 | cosine-similarity: 0.5759
Pagina 13 | cosine-similarity: 0.5444
Pagina 23 | cosine-similarity: 0.5252
Pagina 22 | cosine-similarity: 0.5219
Pagina 27 | cosine-similarity: 0.5150
Pagina 14 | cosine-similarity: 0.5089
Pagina 14 | cosine-similarity: 0.4979
Pagina 15 | cosine-similarity: 0.4949

Antwoord:

Stage is mogelijk bij een bedrijf. Het is mogelijk zelf opdrachten voor te dragen als stageopdracht. Een stage kan ook gevolgd worden in het buitenland. Het is toegestaan om zowel stage in het binnenland als in het buitenland te volgen. Dit zijn verschillende semesters.


**Observaties**

Zoals we hierboven kunnen afleiden, geeft het model vrij goed antwoord op de vraag! De cosine-similarities sluiten doorgaans ook nog eens goed aan! 

Nu willen we graag adequaat meerdere vragen stellen om te testen of het echt goed werkt. 

We zagen initieel dat het lang duurde met 8 chunks, dus hebben we voor het gemak 4 gebruikt. De antwoorden zijn vooralsnog prima! 

Hieronder hebben we dezelfde code maar gebruiken we een array en een loop om door alle vragen heen te gaan. 

Ethiek: terugkerend op voorgaande vraag; hoe kunnen wij discrimineren op cosine similarities terwijl dat niet altijd doorslaggevend kan zijn? In hoeverre is dat ethisch verantwoord?

In [6]:
#array en loop

vragen = [
    "Kan ik stage lopen?",
    "Wanneer zijn er toetsmomenten?",
    "Wat doet een studentenbegeleider?"
]

#looping
for nummer, vraag in enumerate(vragen, start=1):
    print(f"\n{'=' * 60}\nVraag {nummer}: {vraag}")

    # 1. Retrieven: dezelfde methode wordt voor iedere vraag gebruikt.
    vraag_embedding = embedding_model.encode([vraag])[0].tolist()
    resultaat = collection.query(
        query_embeddings=[vraag_embedding],
        n_results=5,
    )
    bronnen = [
        {"tekst": tekst, "pagina": metadata["page"]}
        for tekst, metadata in zip(
            resultaat["documents"][0], resultaat["metadatas"][0]
        )
    ]
    for bron, afstand in zip(bronnen, resultaat["distances"][0]):
        print(f"Pagina {bron['pagina']} | cosine-similarity: {1 - afstand:.4f}")

    # 2. Augmenteren: voeg de gevonden bronnen toe aan de prompt.
    context = "\n\n".join(
        f"[Pagina {bron['pagina']}] {bron['tekst']}" for bron in bronnen
    )
    prompt = f"""Beantwoord de vraag over het OER in duidelijk Nederlands.
Gebruik uitsluitend de context, geef maximaal vier zinnen en vermeld de bronpagina's.
Neem voorwaarden en uitzonderingen mee. Verzin niets.
Als het antwoord niet in de context staat, zeg dan exact: "Dit kan ik niet vinden in het OER."

Context:
{context}

Vraag: {vraag}"""

    # 3. Genereren: dezelfde Qwen-aanroep wordt voor iedere vraag gebruikt.
    ollama_request = Request(
        "http://localhost:11434/api/chat",
        data=json.dumps({
            "model": "qwen2.5:3b",
            "messages": [
                {"role": "system", "content": "Je bent een nauwkeurige Nederlandstalige OER-assistent. Gebruik uitsluitend de context."},
                {"role": "user", "content": prompt},
            ],
            "stream": False,
            "options": {"temperature": 0.1},
        }).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urlopen(ollama_request, timeout=300) as response:
        antwoord = json.loads(response.read().decode("utf-8"))["message"]["content"].strip()
    print(f"\nAntwoord:\n{antwoord}")



Vraag 1: Kan ik stage lopen?
Pagina 17 | cosine-similarity: 0.3526
Pagina 28 | cosine-similarity: 0.3473
Pagina 22 | cosine-similarity: 0.3180
Pagina 14 | cosine-similarity: 0.3122
Pagina 14 | cosine-similarity: 0.3060

Antwoord:
Ja, u kunt stage lopen. Het OER toont dat het mogelijk is om een semester (30EC) stage te lopen bij een bedrijf. Tijdens een stage voert de student een opdracht uit op de werkplek. Begeleiding vindt plaats door het stagebedrijf. Het is mogelijk zelf opdrachten voor te dragen als stageopdracht.

Vraag 2: Wanneer zijn er toetsmomenten?
Pagina 28 | cosine-similarity: 0.6881
Pagina 29 | cosine-similarity: 0.6288
Pagina 28 | cosine-similarity: 0.5886
Pagina 29 | cosine-similarity: 0.5746
Pagina 14 | cosine-similarity: 0.5638

Antwoord:
Er zijn toetsmomenten in verschillende periodes van het studiejaar. De belangrijkste periode is de toetsweek van 28 juni tot 2 juli 2027, die is aangegeven op pagina 28. Dit is de toetsperiode voor de studie in het semester 2 (1 feb

Ethiek: in hoeverre is het ethisch verantwoord om bepaalde chunks weg te laten waardoor de chatbot niet het volledig beeld heeft?

**Observaties**

Met het bijstellen van de chunks en prompt aanpassen zie we dat het antwoord toch nauwkeuriger word. Dat zien we graag terug!

Als we de bepaalde chunks aanpassen per vraag dan zien we dat het antwoord daarin tegen ook beter wordt. Een lage cosine-similarity geeft verder ook niet een representatie of een antwoord nou wel of niet goed antwoord. Het antwoord zelf is hier leidend en dat moet menselijk dus beoordeelt worden.

De antwoorden op de vragen zijn accuraat. Wel zie ik een grammaticale fout bij "een studentenbegeleider doet..." dat moet "voert" zijn. Verder is het antwoord logisch opgebouwd en geeft het adequaat antwoord.

We hebben verder ook de system prompt aangepast. Er zijn reeds twee prompts: de user prompt en system prompt. Bij het adequaat aanpassen van beide zagen we dat het model ook veel beter antwoord op de vragen geven. 

Daarvoor hebben wij dus geen fine-tuning voor nodig. Fine-tuning lijkt mij meer voor trainen van foto's teksten en dergelijke. 


# Deployment



Nu we een goed model hebben die adequaat antwoord kan geven moet er nog gekeken worden naar een user interface.

We zullen een python functie maken waarbij de vraag als een soort input vanuit de browser komt. Via de webserver wordt de python functie aangeroepen waarmee de vragen correct worden beantwoord.

We zullen ons limiteren op vanilla html/javascript/css stack. Geen front-end/backend frameworks worden hier gebruikt.

In [ ]:
#website maken waarin de chat beschikbaar staat.

import json
import threading
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from urllib.request import Request, urlopen

TOP_K = 8

#cosine similarity bekijken in de console

def cosine_similarity(left, right):
    return float(left @ right / ((left @ left) ** 0.5 * (right @ right) ** 0.5))

#functie om de vraag te stellen aan AI.
def beantwoord_vraag(vraag):
    vraag_embedding = embedding_model.encode([vraag])[0]
    bronnen = sorted(
        [
            {
                "tekst": chunk["text"],
                "pagina": chunk["page"],
                "cosine_similarity": round(cosine_similarity(vraag_embedding, embedding), 4),
            }
            for chunk, embedding in zip(chunks, embeddings)
        ],
        key=lambda bron: bron["cosine_similarity"],
        reverse=True,
    )[:TOP_K]
    context = "\n\n".join(f"[Pagina {bron['pagina']}] {bron['tekst']}" for bron in bronnen)
    prompt = f"""Beantwoord de vraag over het OER in duidelijk Nederlands.
Gebruik uitsluitend de context, geef maximaal vier zinnen en vermeld de bronpagina's.
Neem voorwaarden en uitzonderingen mee. Verzin niets.
Als het antwoord niet in de context staat, zeg dan exact: "Dit kan ik niet vinden in het OER."

Context:
{context}

#gebruik lokaal model
Vraag: {vraag}"""
    request = Request(
        "http://localhost:11434/api/chat",
        data=json.dumps({
            "model": "qwen2.5:3b",
            "messages": [
                {"role": "system", "content": "Je bent een nauwkeurige Nederlandstalige OER-assistent. Gebruik uitsluitend de context."},
                {"role": "user", "content": prompt},
            ],
            "stream": False,
            "options": {"temperature": 0.1},
        }).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urlopen(request, timeout=300) as response:
        antwoord = json.loads(response.read().decode("utf-8"))
    return {"answer": antwoord["message"]["content"].strip(), "retrieval": bronnen}
#handler om vragen te beantwoorden
class ChatHandler(SimpleHTTPRequestHandler):
    def _headers(self):
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Headers", "Content-Type")
        self.send_header("Access-Control-Allow-Methods", "POST, OPTIONS")

    def do_OPTIONS(self):
        self.send_response(204)
        self._headers()
        self.end_headers()

    def do_POST(self):
        if self.path != "/api/chat":
            self.send_error(404)
            return
        length = int(self.headers.get("Content-Length", "0"))
        payload = json.loads(self.rfile.read(length))
        body = json.dumps(beantwoord_vraag(payload["question"]), ensure_ascii=False).encode("utf-8")
        self.send_response(200)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self._headers()
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

try:
    server.shutdown()
    server.server_close()
except NameError:
    pass
server = ThreadingHTTPServer(("localhost", 8000), ChatHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()
print("Chatbot beschikbaar op http://localhost:8000")


Chatbot beschikbaar op http://localhost:8000


127.0.0.1 - - [17/Nov/2026 17:57:53] "GET / HTTP/1.1" 304 -
127.0.0.1 - - [17/Nov/2026 17:57:53] "GET /style.css?v=windesheim-2 HTTP/1.1" 304 -
127.0.0.1 - - [17/Nov/2026 17:58:14] "POST /api/chat HTTP/1.1" 200 -
127.0.0.1 - - [17/Nov/2026 17:58:53] "POST /api/chat HTTP/1.1" 200 -
127.0.0.1 - - [17/Nov/2026 17:59:26] "POST /api/chat HTTP/1.1" 200 -


Ethiek: in hoeverre is het ethisch verantwoord een chatbot online beschikbaar te stellen terwijl er enkel middels interne documenten worden gehandeld?

# Conclusie

Op basis van de uitgevoerde testen is de chatbot als prototype succesvol. De geteste vragen over stage, toetsmomenten en studentenbegeleiding kregen inhoudelijk passende antwoorden uit het OER. De combinatie van het meertalige embeddingmodel, relevante tekstfragmenten en de aangescherpte system- en userprompt heeft de kwaliteit van de antwoorden duidelijk verbeterd.

## Beantwoording van de KPI's 

- **Minimaal 80% correcte antwoorden:** voorlopig behaald in de handmatige test.
- **Minimaal 4 van de 5 opgehaalde fragmenten relevant:** grotendeels behaald. De opgehaalde pagina's sluiten bij de voorbeeldvragen aan, maar het aantal relevante fragmenten is niet per vraag formeel geteld. Hiervoor is een grotere, gelabelde testset nodig.
- **Antwoorden bevatten een bronpagina:** gedeeltelijk behaald. De retrieval-resultaten bewaren en tonen de paginanummers. De prompt vraagt het model ook om bronpagina's te vermelden, maar dit gebeurt nog niet in ieder getoond antwoord even duidelijk.
- **De chatbot verzint geen informatie:** in de uitgevoerde voorbeelden behaald. 
- **Een antwoord verschijnt binnen 60 seconden:** in de uitgevoerde lokale tests lijkt deze KPI behaald.

## Beantwoording van de KSF's

- Het OER is ingelezen en verwerkt tot 77 tekstchunks, embeddings en ChromaDB-items. Deze KSF is behaald.
- Het model antwoordt in de uitgevoerde voorbeelden op basis van de meegegeven context. De systemprompt en userprompt ondersteunen dit gedrag. Deze KSF is behaald voor het geteste prototype.
- De bronverwijzing is technisch beschikbaar via de paginametadata en de retrieval-uitvoer. De presentatie daarvan in elk eindantwoord kan nog consistenter. Deze KSF is gedeeltelijk behaald.
- De chatbot is lokaal beschikbaar via de deployment op `http://localhost:8000` en werkte tijdens de demo. Deze KSF is behaald voor de huidige lokale demonstratie.

## Randvoorwaarde

De randvoorwaarde is dat de context per vraag maximaal 1500 tokens bevat. Bij het handmatig runnen van een model heb je eigenlijk unlimited tokens. Maar gebruik je een zwaar model dan is het handig om token limit aan te geven. Dat was hier dus niet nodig!


